# Applied Data Mining with Python

## Part 2: Feature Engineering & Classical Models

*dr. Uroš Godnov*  ·  Notebook companion to the Quarto slide deck.

> This notebook runs top-to-bottom on Google Colab's free tier. Each **Hands-on Exercise** has a `# TODO` cell to complete, followed by a collapsible **💡 Show solution**. Run the setup cells in order first.

In [ ]:
# --- Robust data loader (added for the notebook version) -------------------------------
# Loads the wine CSV from the UCI repository, falling back to a GitHub mirror if UCI is
# unreachable during a live session. Used everywhere the deck wrote pd.read_csv(url, ...).
import pandas as pd

def load_wine(url, sep=";"):
    try:
        return pd.read_csv(url, sep=sep)
    except Exception as e:
        fname = url.rsplit("/", 1)[-1]
        mirror = "https://raw.githubusercontent.com/zygmuntz/wine-quality/master/winequality/" + fname
        print(f"Primary source failed ({type(e).__name__}); falling back to mirror.")
        return pd.read_csv(mirror, sep=sep)

## Part 2 Overview

**Feature Engineering & Classical Models** (≈ 3.5 hours)

| Module | Topic | Duration |
|--------|-------|----------|
| 2.1 | Feature Engineering with Impact | 40 min |
| 2.2 | Feature Selection Methods | 30 min |
| 2.3 | Train/Test Split and Validation | 25 min |
| 2.4 | Overfitting vs Underfitting | 25 min |
| 2.5 | Linear Regression | 30 min |
| 2.6 | Logistic Regression | 25 min |
| 2.7 | Lasso and Ridge Regression | 30 min |
| 2.8 | Wow Moment: Performance Improvements | 15 min |

## Learning Objectives

By the end of this session, you will be able to:

- Engineer features that improve model performance
- Apply feature selection to reduce dimensionality
- Properly split data for training and evaluation
- Diagnose and address overfitting and underfitting
- Implement and interpret linear, logistic, lasso, and ridge regression
- Measure and compare model performance

## Setup: Loading Our Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load wine quality dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = load_wine(url)

print(f"Dataset loaded: {wine_df.shape[0]} rows, {wine_df.shape[1]} columns")
print(f"Target variable: quality (range {wine_df['quality'].min()}-{wine_df['quality'].max()})")

# Module 2.1: Feature Engineering with Impact

## Conceptual Overview

Feature engineering transforms raw data into features that better represent the underlying patterns. Good features can dramatically improve model performance, often more than algorithm selection.

Types of feature engineering:

- Mathematical transformations (log, polynomial)
- Domain-specific combinations
- Encoding categorical variables
- Creating interaction terms

## Wow Factor Hook

You will see a baseline model's R² improve by over 15% through strategic feature engineering alone — no algorithm change required.

## Code: Baseline Model Performance

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare baseline features
X_baseline = wine_df.drop('quality', axis=1)
y = wine_df['quality']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline linear regression
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_baseline = baseline_model.predict(X_test_scaled)
baseline_r2 = r2_score(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print("BASELINE MODEL PERFORMANCE")
print("=" * 40)
print(f"R² Score: {baseline_r2:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"\nThis is our benchmark to beat with feature engineering.")

## Code: Polynomial Features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_baseline)

print(f"Original features: {X_baseline.shape[1]}")
print(f"Polynomial features: {X_poly.shape[1]}")

# Get feature names
feature_names = poly.get_feature_names_out(X_baseline.columns)
print(f"\nSample new features:")
for name in feature_names[11:20]:  # Show some interaction terms
    print(f"  - {name}")

## Code: Domain-Specific Feature Engineering

In [ ]:
# Create engineered features dataframe
wine_engineered = wine_df.copy()

# Ratio features (chemically meaningful)
wine_engineered['free_sulfur_ratio'] = (
    wine_engineered['free sulfur dioxide'] / 
    (wine_engineered['total sulfur dioxide'] + 0.001)  # avoid division by zero
)

wine_engineered['acid_ratio'] = (
    wine_engineered['fixed acidity'] / 
    (wine_engineered['volatile acidity'] + 0.001)
)

wine_engineered['sugar_acid_ratio'] = (
    wine_engineered['residual sugar'] / 
    (wine_engineered['fixed acidity'] + 0.001)
)

# Combined acidity
wine_engineered['total_acidity'] = (
    wine_engineered['fixed acidity'] + 
    wine_engineered['volatile acidity'] + 
    wine_engineered['citric acid']
)

# Alcohol-density interaction (affects mouthfeel)
wine_engineered['alcohol_density'] = (
    wine_engineered['alcohol'] * wine_engineered['density']
)

# Log transformations for skewed features
wine_engineered['log_chlorides'] = np.log1p(wine_engineered['chlorides'])
wine_engineered['log_sulfur'] = np.log1p(wine_engineered['total sulfur dioxide'])

print("New engineered features:")
new_cols = ['free_sulfur_ratio', 'acid_ratio', 'sugar_acid_ratio', 
            'total_acidity', 'alcohol_density', 'log_chlorides', 'log_sulfur']
print(wine_engineered[new_cols].describe().round(3))

## Code: Evaluating Engineered Features

In [ ]:
# Prepare engineered features
X_engineered = wine_engineered.drop('quality', axis=1)
y = wine_engineered['quality']

# Split and scale
X_train_eng, X_test_eng, y_train, y_test = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42
)

scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

# Train model with engineered features
eng_model = LinearRegression()
eng_model.fit(X_train_eng_scaled, y_train)

# Evaluate
y_pred_eng = eng_model.predict(X_test_eng_scaled)
eng_r2 = r2_score(y_test, y_pred_eng)
eng_rmse = np.sqrt(mean_squared_error(y_test, y_pred_eng))

print("ENGINEERED FEATURES MODEL PERFORMANCE")
print("=" * 40)
print(f"R² Score: {eng_r2:.4f} (baseline: {baseline_r2:.4f})")
print(f"RMSE: {eng_rmse:.4f} (baseline: {baseline_rmse:.4f})")
print(f"\nImprovement in R²: {((eng_r2 - baseline_r2) / baseline_r2 * 100):.1f}%")

## Code: Feature Importance After Engineering

In [ ]:
# Get feature importances from coefficients
feature_importance = pd.DataFrame({
    'feature': X_engineered.columns,
    'coefficient': eng_model.coef_,
    'abs_coefficient': np.abs(eng_model.coef_)
}).sort_values('abs_coefficient', ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
colors = ['green' if x > 0 else 'red' for x in top_features['coefficient']]
plt.barh(top_features['feature'], top_features['coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.title('Top 15 Feature Importances (Linear Regression Coefficients)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 most important features:")
print(feature_importance[['feature', 'coefficient']].head(10).to_string(index=False))

## Expected Output & Interpretation

Feature engineering results:

- **Baseline R²**: ~0.36
- **Engineered R²**: ~0.42 (approximately 16% improvement)
- **Top engineered features**: acid_ratio, alcohol_density show strong coefficients
- **Insight**: Domain knowledge helps create predictive features

## Hands-on Exercise 2.1

**Task**: Create your own engineered features.

1. Create a feature `sulfur_balance = free sulfur dioxide - total sulfur dioxide`
2. Create a squared feature `alcohol_squared = alcohol ** 2`
3. Create an interaction `ph_alcohol = pH * alcohol`
4. Train a linear regression with these new features added
5. Compare R² to the baseline — did your features help?

**Time**: 15 minutes

In [ ]:
# TODO: engineer sulfur_balance, alcohol_squared, ph_alcohol; train LinearRegression; compare R^2 to baseline
# Hint: build on a copy of wine_df, then scale -> fit -> r2_score on the test split


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

we = wine_df.copy()
we["sulfur_balance"] = we["free sulfur dioxide"] - we["total sulfur dioxide"]
we["alcohol_squared"] = we["alcohol"] ** 2
we["ph_alcohol"] = we["pH"] * we["alcohol"]

X = we.drop("quality", axis=1); y = we["quality"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
m = LinearRegression().fit(sc.fit_transform(X_tr), y_tr)
print(f"R2 with new features: {r2_score(y_te, m.predict(sc.transform(X_te))):.4f}")
print("Compare against the engineered baseline (~0.40) to see if these features helped.")
```

</details>

# Module 2.2: Feature Selection Methods

## Conceptual Overview

Not all features improve predictions. Feature selection identifies the most relevant features by:

- Removing redundant information
- Reducing overfitting risk
- Improving model interpretability
- Decreasing training time

Methods include filter (correlation), wrapper (recursive elimination), and embedded (regularization) approaches.

## Wow Factor Hook

You will reduce the feature set by half while maintaining nearly identical performance — proving that simpler models can match complex ones.

## Code: Correlation-Based Selection (Filter Method)

In [ ]:
# Calculate correlation with target
correlations = wine_engineered.corr(numeric_only=True)['quality'].drop('quality').abs().sort_values(ascending=False)

print("Feature correlations with quality:")
print(correlations.round(3))

# Select features with correlation > threshold
threshold = 0.15
selected_corr_features = correlations[correlations > threshold].index.tolist()
print(f"\nFeatures with |correlation| > {threshold}: {len(selected_corr_features)}")
print(selected_corr_features)

# Visualize
plt.figure(figsize=(12, 6))
correlations.plot(kind='bar', color=['green' if x > threshold else 'lightgray' for x in correlations])
plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold = {threshold}')
plt.title('Feature Correlations with Quality')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## Code: Recursive Feature Elimination (Wrapper Method)

In [ ]:
from sklearn.feature_selection import RFE

# Prepare data
X = wine_engineered.drop('quality', axis=1)
y = wine_engineered['quality']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# RFE with linear regression
model = LinearRegression()
rfe = RFE(estimator=model, n_features_to_select=10, step=1)
rfe.fit(X_scaled, y)

# Get selected features
rfe_selected = X.columns[rfe.support_].tolist()
rfe_ranking = pd.DataFrame({
    'feature': X.columns,
    'ranking': rfe.ranking_,
    'selected': rfe.support_
}).sort_values('ranking')

print("RFE Feature Selection Results:")
print("=" * 40)
print(f"Selected features ({len(rfe_selected)}):")
for feat in rfe_selected:
    print(f"  - {feat}")
    
print("\nFull ranking:")
print(rfe_ranking.head(15).to_string(index=False))

## Code: Variance Threshold Selection

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Remove low-variance features (after scaling)
selector = VarianceThreshold(threshold=0.1)
X_high_variance = selector.fit_transform(X_scaled)

# Get selected feature names
variance_mask = selector.get_support()
selected_var_features = X.columns[variance_mask].tolist()

print(f"Original features: {X_scaled.shape[1]}")
print(f"After variance threshold: {X_high_variance.shape[1]}")
print(f"Removed: {X_scaled.shape[1] - X_high_variance.shape[1]} low-variance features")

# Show variances
variances = pd.DataFrame({
    'feature': X.columns,
    'variance': selector.variances_,
    'selected': variance_mask
}).sort_values('variance', ascending=False)
print("\nFeature variances (scaled data):")
print(variances.head(10).to_string(index=False))

## Code: Comparing Selection Methods

In [ ]:
# Evaluate models with different feature sets
results = []

# Full feature set
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
model_full = LinearRegression()
model_full.fit(X_train_full, y_train)
r2_full = r2_score(y_test, model_full.predict(X_test_full))
results.append(('All Features', X.shape[1], r2_full))

# Correlation-selected features
X_corr = wine_engineered[selected_corr_features]
X_corr_scaled = StandardScaler().fit_transform(X_corr)
X_train_corr, X_test_corr, _, _ = train_test_split(
    X_corr_scaled, y, test_size=0.2, random_state=42
)
model_corr = LinearRegression()
model_corr.fit(X_train_corr, y_train)
r2_corr = r2_score(y_test, model_corr.predict(X_test_corr))
results.append(('Correlation Filter', len(selected_corr_features), r2_corr))

# RFE-selected features
X_rfe = wine_engineered[rfe_selected]
X_rfe_scaled = StandardScaler().fit_transform(X_rfe)
X_train_rfe, X_test_rfe, _, _ = train_test_split(
    X_rfe_scaled, y, test_size=0.2, random_state=42
)
model_rfe = LinearRegression()
model_rfe.fit(X_train_rfe, y_train)
r2_rfe = r2_score(y_test, model_rfe.predict(X_test_rfe))
results.append(('RFE Selection', len(rfe_selected), r2_rfe))

# Summary table
results_df = pd.DataFrame(results, columns=['Method', 'Features', 'R² Score'])
print("\nFeature Selection Comparison:")
print("=" * 50)
print(results_df.to_string(index=False))

## Expected Output & Interpretation

Feature selection findings:

- **All features**: R² ~0.42 with 18 features
- **Correlation filter**: R² ~0.40 with 12 features (7% drop in R², 33% fewer features)
- **RFE selection**: R² ~0.41 with 10 features (minimal drop, nearly half the features)

Fewer features with similar performance means a more parsimonious, interpretable model.

## Hands-on Exercise 2.2

**Task**: Implement your own feature selection pipeline.

1. Use SelectKBest from sklearn with f_regression as the scoring function
2. Select the top 8 features
3. Train a linear regression model with only these 8 features
4. Compare the R² score to the full feature set
5. Which features were selected?

**Time**: 10 minutes

In [ ]:
# TODO: SelectKBest(f_regression, k=8) on wine_engineered, train LR on the 8, compare R^2, list selected
# Hint: from sklearn.feature_selection import SelectKBest, f_regression


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

X = wine_engineered.drop("quality", axis=1); y = wine_engineered["quality"]

sel = SelectKBest(score_func=f_regression, k=8).fit(X, y)
selected = X.columns[sel.get_support()].tolist()
print("Selected features:", selected)

Xk = X[selected]
X_tr, X_te, y_tr, y_te = train_test_split(Xk, y, test_size=0.2, random_state=42)
sc = StandardScaler()
m = LinearRegression().fit(sc.fit_transform(X_tr), y_tr)
print(f"R2 with top 8 features: {r2_score(y_te, m.predict(sc.transform(X_te))):.4f}")
```

</details>

# Module 2.3: Train/Test Split and Validation

## Conceptual Overview

Proper data splitting is essential for unbiased model evaluation:

- **Training set**: Used to fit the model
- **Validation set**: Used for hyperparameter tuning
- **Test set**: Final, unbiased performance estimate

Never evaluate your model on data it was trained on — this leads to overly optimistic results.

## Wow Factor Hook

You will see how improper splitting leads to misleading performance metrics and learn to avoid this critical mistake.

## Code: Basic Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Load fresh data
X = wine_df.drop('quality', axis=1)
y = wine_df['quality']

# Standard 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data Split Summary:")
print("=" * 40)
print(f"Total samples: {len(X)}")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test samples: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

# Check target distribution is preserved (stratification check)
print(f"\nTarget distribution:")
print(f"Full data mean: {y.mean():.3f}")
print(f"Training mean: {y_train.mean():.3f}")
print(f"Test mean: {y_test.mean():.3f}")

## Code: Stratified Splitting

In [ ]:
# Compare stratified vs non-stratified splits
# Non-stratified
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None
)

# Stratified
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].bar(y.value_counts().sort_index().index, 
            y.value_counts().sort_index().values / len(y))
axes[0].set_title('Original Distribution')
axes[0].set_xlabel('Quality')
axes[0].set_ylabel('Proportion')

# Non-stratified test set
axes[1].bar(y_test_ns.value_counts().sort_index().index, 
            y_test_ns.value_counts().sort_index().values / len(y_test_ns))
axes[1].set_title('Non-Stratified Test Set')
axes[1].set_xlabel('Quality')

# Stratified test set
axes[2].bar(y_test_s.value_counts().sort_index().index, 
            y_test_s.value_counts().sort_index().values / len(y_test_s))
axes[2].set_title('Stratified Test Set')
axes[2].set_xlabel('Quality')

plt.tight_layout()
plt.show()

print("Quality distribution comparison (proportions):")
comparison = pd.DataFrame({
    'Original': y.value_counts(normalize=True).sort_index(),
    'Non-Stratified Test': y_test_ns.value_counts(normalize=True).sort_index(),
    'Stratified Test': y_test_s.value_counts(normalize=True).sort_index()
}).round(3)
print(comparison)

## Code: The Data Leakage Problem

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# WRONG: Scaling before split (data leakage)
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)  # Fitted on ALL data
X_train_wrong, X_test_wrong, y_train, y_test = train_test_split(
    X_scaled_wrong, y, test_size=0.2, random_state=42
)

model_wrong = LinearRegression()
model_wrong.fit(X_train_wrong, y_train)
r2_wrong = r2_score(y_test, model_wrong.predict(X_test_wrong))

# CORRECT: Scaling after split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train)  # Fit only on training
X_test_correct = scaler_correct.transform(X_test)  # Transform test using training params

model_correct = LinearRegression()
model_correct.fit(X_train_correct, y_train)
r2_correct = r2_score(y_test, model_correct.predict(X_test_correct))

print("Data Leakage Demonstration:")
print("=" * 50)
print(f"WRONG (scaling before split): R² = {r2_wrong:.4f}")
print(f"CORRECT (scaling after split): R² = {r2_correct:.4f}")
print(f"\nDifference: {abs(r2_wrong - r2_correct):.4f}")
print("\nNote: The 'wrong' method may show similar or slightly better")
print("results, but it's fundamentally flawed and unreliable.")

## Code: Train/Validation/Test Split

In [ ]:
# Three-way split: 60% train, 20% validation, 20% test

# First split: separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: separate validation from training
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42  # 0.25 * 0.8 = 0.2
)

print("Three-Way Split:")
print("=" * 40)
print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")

# Scale properly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nAll sets scaled using training set parameters only.")

## Expected Output & Interpretation

Key takeaways:

- **Stratified splitting** preserves class distributions, critical for imbalanced data
- **Data leakage** can artificially inflate performance metrics
- **Three-way split** enables hyperparameter tuning without contaminating test evaluation

## Hands-on Exercise 2.3

**Task**: Implement a proper splitting workflow.

1. Perform a stratified 70/15/15 split (train/val/test)
2. Scale features using only training data statistics
3. Train a linear regression on the training set
4. Report R² on validation and test sets separately
5. Verify the target distribution is similar across all three sets

**Time**: 10 minutes

In [ ]:
# TODO: stratified 70/15/15 split, scale on train only, report val & test R^2, check target means
# Hint: two train_test_split calls with stratify=; second test_size ~0.1765 of the 85% remainder


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = wine_df.drop("quality", axis=1); y = wine_df["quality"]

# 70 / 15 / 15 stratified split
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=42, stratify=y_tmp)  # 0.1765*0.85 ~ 0.15

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_val_s = sc.transform(X_val)
X_test_s = sc.transform(X_test)

m = LinearRegression().fit(X_train_s, y_train)
print(f"Validation R2: {r2_score(y_val, m.predict(X_val_s)):.4f}")
print(f"Test R2:       {r2_score(y_test, m.predict(X_test_s)):.4f}")
print(f"Target means -> train {y_train.mean():.3f}, val {y_val.mean():.3f}, test {y_test.mean():.3f}")
```

</details>

# Module 2.4: Overfitting vs Underfitting

## Conceptual Overview

Model complexity must match data complexity:

- **Underfitting**: Model too simple, high bias, poor performance on both train and test
- **Overfitting**: Model too complex, high variance, great train performance, poor test performance
- **Good fit**: Balanced complexity, similar performance on train and test

The goal is to find the sweet spot where the model generalizes well.

## Wow Factor Hook

You will visualize the bias-variance tradeoff with polynomial regression, seeing exactly how model complexity affects generalization.

## Code: Generating Demonstration Data

In [ ]:
# Create synthetic data with known relationship
np.random.seed(42)
n_samples = 100

X_demo = np.linspace(0, 10, n_samples).reshape(-1, 1)
y_true = 2 + 0.5 * X_demo.ravel() + 0.3 * X_demo.ravel()**2 - 0.02 * X_demo.ravel()**3
y_demo = y_true + np.random.normal(0, 2, n_samples)  # Add noise

# Split
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_demo, y_demo, test_size=0.3, random_state=42
)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_train_demo, y_train_demo, alpha=0.7, label='Training data', color='blue')
plt.scatter(X_test_demo, y_test_demo, alpha=0.7, label='Test data', color='red')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Demonstration Dataset')
plt.legend()
plt.show()

## Code: Visualizing Underfitting and Overfitting

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Train models with different complexities
degrees = [1, 3, 15]
titles = ['Underfitting (degree=1)', 'Good Fit (degree=3)', 'Overfitting (degree=15)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

X_plot = np.linspace(0, 10, 100).reshape(-1, 1)

for idx, degree in enumerate(degrees):
    ax = axes[idx]
    
    # Create polynomial model
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    model.fit(X_train_demo, y_train_demo)
    
    # Predictions
    y_train_pred = model.predict(X_train_demo)
    y_test_pred = model.predict(X_test_demo)
    y_plot = model.predict(X_plot)
    
    # Metrics
    train_r2 = r2_score(y_train_demo, y_train_pred)
    test_r2 = r2_score(y_test_demo, y_test_pred)
    
    # Plot
    ax.scatter(X_train_demo, y_train_demo, alpha=0.6, label='Train', color='blue')
    ax.scatter(X_test_demo, y_test_demo, alpha=0.6, label='Test', color='red')
    ax.plot(X_plot, y_plot, 'g-', linewidth=2, label='Model')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'{titles[idx]}\nTrain R²={train_r2:.3f}, Test R²={test_r2:.3f}')
    ax.legend()
    ax.set_ylim(-5, 25)

plt.tight_layout()
plt.show()

## Code: Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve

# Learning curves for different model complexities
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, degree in enumerate([1, 3, 15]):
    ax = axes[idx]
    
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    
    train_sizes, train_scores, test_scores = learning_curve(
        model, X_demo, y_demo,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5,
        scoring='r2',
        random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    test_mean = test_scores.mean(axis=1)
    test_std = test_scores.std(axis=1)
    
    ax.plot(train_sizes, train_mean, 'b-', label='Training score')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.plot(train_sizes, test_mean, 'r-', label='Cross-validation score')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='red')
    
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('R² Score')
    ax.set_title(f'Learning Curve (degree={degree})')
    ax.legend(loc='lower right')
    ax.set_ylim(-0.5, 1.1)

plt.tight_layout()
plt.show()

## Code: Validation Curve

In [ ]:
from sklearn.model_selection import validation_curve

# Validation curve: how test performance changes with model complexity
degrees = range(1, 16)
train_scores_list = []
test_scores_list = []

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    
    train_scores, test_scores = validation_curve(
        model, X_demo, y_demo,
        param_name='polynomialfeatures__degree',
        param_range=[degree],
        cv=5,
        scoring='r2'
    )
    
    train_scores_list.append(train_scores.mean())
    test_scores_list.append(test_scores.mean())

# Plot validation curve
plt.figure(figsize=(10, 6))
plt.plot(degrees, train_scores_list, 'b-o', label='Training score', linewidth=2)
plt.plot(degrees, test_scores_list, 'r-o', label='Cross-validation score', linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('Validation Curve: Model Complexity vs Performance')
plt.legend()
plt.axvline(x=3, color='green', linestyle='--', label='Optimal complexity')
plt.xticks(degrees)
plt.grid(True, alpha=0.3)
plt.show()

print("Interpretation:")
print("- Low degree: Both scores low (underfitting)")
print("- Optimal degree (~3): Both scores high, gap is small (good fit)")
print("- High degree: Train score high, test score drops (overfitting)")

## Expected Output & Interpretation

Bias-variance observations:

- **Degree 1 (underfit)**: Low R² on both sets, model misses the pattern
- **Degree 3 (good fit)**: Similar R² on train and test, captures the true relationship
- **Degree 15 (overfit)**: Near-perfect train R², poor test R², memorized noise

## Hands-on Exercise 2.4

**Task**: Diagnose overfitting on real data.

1. Using the wine dataset, train polynomial regression models with degrees 1, 2, 3, and 5
2. For each model, report train and test R² scores
3. Identify at which degree overfitting begins (train >> test)
4. Plot the train vs test scores as a function of polynomial degree

**Time**: 15 minutes

In [ ]:
# TODO: polynomial degrees 1,2,3,5 on wine; record train vs test R^2; plot; find where overfitting starts
# Hint: make_pipeline(StandardScaler(), PolynomialFeatures(d), LinearRegression())


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

X = wine_df.drop("quality", axis=1); y = wine_df["quality"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

degrees = [1, 2, 3, 5]; train_r2 = []; test_r2 = []
for d in degrees:
    mp = make_pipeline(StandardScaler(), PolynomialFeatures(d), LinearRegression())
    mp.fit(X_tr, y_tr)
    train_r2.append(r2_score(y_tr, mp.predict(X_tr)))
    test_r2.append(r2_score(y_te, mp.predict(X_te)))
    print(f"degree {d}: train R2={train_r2[-1]:.3f}, test R2={test_r2[-1]:.3f}")

plt.plot(degrees, train_r2, "b-o", label="Train")
plt.plot(degrees, test_r2, "r-o", label="Test")
plt.xlabel("Polynomial degree"); plt.ylabel("R2"); plt.legend()
plt.title("Overfitting on wine data"); plt.show()
print("Overfitting begins where train R2 keeps rising while test R2 drops (usually degree >= 3).")
```

</details>

# Module 2.5: Linear Regression

## Conceptual Overview

Linear regression models the relationship between features and a continuous target as a weighted sum:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

Key properties:

- Interpretable coefficients
- Fast training
- Assumes linear relationships
- Sensitive to outliers and multicollinearity

## Wow Factor Hook

You will build a complete linear regression pipeline on wine quality data and understand exactly how each feature contributes to predictions.

## Code: Complete Linear Regression Pipeline

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Prepare data
X = wine_df.drop('quality', axis=1)
y = wine_df['quality']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = lr_model.predict(X_train_scaled)
y_test_pred = lr_model.predict(X_test_scaled)

# Evaluate
print("LINEAR REGRESSION RESULTS")
print("=" * 50)
print(f"\nTraining Performance:")
print(f"  R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")

print(f"\nTest Performance:")
print(f"  R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

## Code: Coefficient Interpretation

In [ ]:
# Create coefficient summary
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_,
    'Abs_Coefficient': np.abs(lr_model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print(f"Intercept: {lr_model.intercept_:.4f}")
print("\nFeature Coefficients (standardized):")
print(coef_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.xlabel('Coefficient Value (Standardized)')
plt.title('Linear Regression Coefficients')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Positive coefficient: Higher feature value → Higher predicted quality")
print("- Negative coefficient: Higher feature value → Lower predicted quality")
print("- Larger magnitude: Stronger effect on prediction")

## Code: Residual Analysis

In [ ]:
# Calculate residuals
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Residuals vs Predicted
axes[0].scatter(y_test_pred, residuals, alpha=0.5)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Quality')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values')

# Histogram of residuals
axes[1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

# Actual vs Predicted
axes[2].scatter(y_test, y_test_pred, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual Quality')
axes[2].set_ylabel('Predicted Quality')
axes[2].set_title('Actual vs Predicted Values')

plt.tight_layout()
plt.show()

print("Residual Statistics:")
print(f"  Mean: {residuals.mean():.4f} (should be ~0)")
print(f"  Std: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## Expected Output & Interpretation

Linear regression results:

- **R² ~0.36**: Explains about 36% of variance in wine quality
- **Top positive predictors**: alcohol, sulphates
- **Top negative predictors**: volatile acidity, chlorides
- **Residuals**: Centered at zero, approximately normal — good model assumptions

## Hands-on Exercise 2.5

**Task**: Build and interpret a linear regression model.

1. Load the California Housing dataset from sklearn
2. Train a linear regression model (remember to scale features)
3. Report R², RMSE, and MAE on the test set
4. Which feature has the strongest positive effect on house value?
5. Create a residual plot and check if residuals are centered at zero

**Time**: 15 minutes

In [ ]:
# TODO: LinearRegression on California Housing; report R^2/RMSE/MAE; strongest +coef; residual plot
# Hint: from sklearn.datasets import fetch_california_housing


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

cal = fetch_california_housing(as_frame=True)
X, y = cal.data, cal.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)
m = LinearRegression().fit(X_tr_s, y_tr)
pred = m.predict(X_te_s)

print(f"R2={r2_score(y_te, pred):.4f}  "
      f"RMSE={np.sqrt(mean_squared_error(y_te, pred)):.4f}  "
      f"MAE={mean_absolute_error(y_te, pred):.4f}")

coef = pd.Series(m.coef_, index=X.columns).sort_values()
print("Strongest positive effect:", coef.idxmax(), f"({coef.max():.3f})")

res = y_te - pred
plt.scatter(pred, res, alpha=0.2, s=8); plt.axhline(0, color="r", ls="--")
plt.xlabel("Predicted"); plt.ylabel("Residual"); plt.title("Residuals"); plt.show()
print(f"Mean residual: {res.mean():.4f} (should be ~0)")
```

</details>

# Module 2.6: Logistic Regression

## Conceptual Overview

Logistic regression is used for binary classification, not regression. It predicts the probability of belonging to a class:

$$P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ...)}}$$

Output is a probability between 0 and 1, converted to class labels using a threshold (typically 0.5).

## Wow Factor Hook

You will predict whether wine quality is "good" or "not good" with over 75% accuracy, and understand which features drive the classification.

## Code: Binary Classification Setup

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create binary target: good wine (quality >= 7) vs not good
wine_df['good_wine'] = (wine_df['quality'] >= 7).astype(int)

print("Binary target distribution:")
print(wine_df['good_wine'].value_counts())
print(f"\nPercentage of 'good' wines: {wine_df['good_wine'].mean()*100:.1f}%")

# Prepare data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['good_wine']

# Stratified split (important for imbalanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set class distribution:")
print(y_train.value_counts())

## Code: Logistic Regression Training

In [ ]:
# Train logistic regression
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = log_model.predict(X_train_scaled)
y_test_pred = log_model.predict(X_test_scaled)
y_test_prob = log_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print(f"\nTraining Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=['Not Good', 'Good']))

## Code: Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, 
    display_labels=['Not Good', 'Good'],
    ax=axes[0],
    cmap='Blues'
)
axes[0].set_title('Confusion Matrix')

# Probability distribution
axes[1].hist(y_test_prob[y_test == 0], bins=20, alpha=0.7, label='Not Good (actual)', color='red')
axes[1].hist(y_test_prob[y_test == 1], bins=20, alpha=0.7, label='Good (actual)', color='green')
axes[1].axvline(x=0.5, color='black', linestyle='--', label='Decision threshold')
axes[1].set_xlabel('Predicted Probability of Good')
axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## Code: Feature Importance in Logistic Regression

In [ ]:
# Coefficient interpretation (log-odds)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_model.coef_[0],
    'Odds_Ratio': np.exp(log_model.coef_[0])
}).sort_values('Coefficient', ascending=False)

print("Feature Effects on 'Good Wine' Probability:")
print("=" * 50)
print(coef_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.xlabel('Coefficient (Log-Odds)')
plt.title('Logistic Regression Feature Effects')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Positive coefficient: Increases probability of being 'Good'")
print("- Odds ratio > 1: Each unit increase multiplies odds of 'Good'")
print("- Odds ratio < 1: Each unit increase reduces odds of 'Good'")

## Expected Output & Interpretation

Logistic regression results:

- **Accuracy ~75-78%**: Reasonable for imbalanced data
- **Precision vs Recall tradeoff**: High precision for 'Good' class, lower recall
- **Key positive predictors**: alcohol, sulphates
- **Key negative predictors**: volatile acidity, density

## Hands-on Exercise 2.6

**Task**: Experiment with classification thresholds.

1. Using the trained logistic regression, predict probabilities for the test set
2. Try thresholds of 0.3, 0.5, and 0.7 for classifying "good" wines
3. Calculate accuracy, precision, and recall for each threshold
4. Which threshold would you choose if you want to minimize false positives?

**Time**: 10 minutes

In [ ]:
# TODO: sweep thresholds 0.3/0.5/0.7 on the logistic model's probabilities; report acc/precision/recall
# Hint: probs = log_model.predict_proba(X_test_scaled)[:, 1]; pred = (probs >= t).astype(int)


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score

# log_model, X_test_scaled, y_test come from the Module 2.6 teaching cells
probs = log_model.predict_proba(X_test_scaled)[:, 1]
for t in [0.3, 0.5, 0.7]:
    pred = (probs >= t).astype(int)
    print(f"threshold {t}: "
          f"acc={accuracy_score(y_test, pred):.3f}, "
          f"precision={precision_score(y_test, pred, zero_division=0):.3f}, "
          f"recall={recall_score(y_test, pred, zero_division=0):.3f}")
print("To minimize FALSE POSITIVES, pick the higher threshold (0.7): it maximizes precision.")
```

</details>

# Module 2.7: Lasso and Ridge Regression

## Conceptual Overview

Regularized regression adds penalties to prevent overfitting:

- **Ridge (L2)**: Adds squared coefficients penalty; shrinks coefficients but keeps all features
- **Lasso (L1)**: Adds absolute coefficients penalty; can shrink coefficients to exactly zero (feature selection)

Both have a hyperparameter α (alpha) controlling regularization strength.

## Wow Factor Hook

You will see Lasso automatically select the most important features and Ridge prevent overfitting on multicollinear data.

## Code: Comparing Regularization Methods

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Prepare data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'ElasticNet (α=0.1)': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train_scaled))
    test_r2 = r2_score(y_test, model.predict(X_test_scaled))
    n_features = np.sum(model.coef_ != 0)
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Non-zero Features': n_features
    })

results_df = pd.DataFrame(results)
print("Regularization Comparison:")
print("=" * 60)
print(results_df.to_string(index=False))

## Code: Lasso Feature Selection

In [ ]:
# Train Lasso with different alpha values
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
lasso_coefs = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)

# Visualize coefficient paths
plt.figure(figsize=(12, 6))
for i, feature in enumerate(X.columns):
    coef_path = [coefs[i] for coefs in lasso_coefs]
    plt.plot(alphas, coef_path, label=feature, linewidth=2)

plt.xscale('log')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Paths')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Show which features survive at α=0.1
lasso_selected = Lasso(alpha=0.1, max_iter=10000)
lasso_selected.fit(X_train_scaled, y_train)
selected_features = X.columns[lasso_selected.coef_ != 0].tolist()
print(f"\nFeatures selected by Lasso (α=0.1): {len(selected_features)}")
print(selected_features)

## Code: Ridge vs Lasso Coefficient Comparison

In [ ]:
# Compare coefficients across methods
coef_comparison = pd.DataFrame({
    'Feature': X.columns,
    'Linear': LinearRegression().fit(X_train_scaled, y_train).coef_,
    'Ridge': Ridge(alpha=1.0).fit(X_train_scaled, y_train).coef_,
    'Lasso': Lasso(alpha=0.1).fit(X_train_scaled, y_train).coef_
})

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
x = np.arange(len(X.columns))
width = 0.25

ax.bar(x - width, coef_comparison['Linear'], width, label='Linear', alpha=0.8)
ax.bar(x, coef_comparison['Ridge'], width, label='Ridge', alpha=0.8)
ax.bar(x + width, coef_comparison['Lasso'], width, label='Lasso', alpha=0.8)

ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Comparison: Linear vs Ridge vs Lasso')
ax.set_xticks(x)
ax.set_xticklabels(X.columns, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nCoefficient comparison:")
print(coef_comparison.round(4).to_string(index=False))

## Code: Cross-Validated Alpha Selection

In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV

# Cross-validated Ridge
ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print(f"Ridge CV Best Alpha: {ridge_cv.alpha_:.4f}")
print(f"Ridge CV Test R²: {r2_score(y_test, ridge_cv.predict(X_test_scaled)):.4f}")

# Cross-validated Lasso
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
print(f"\nLasso CV Best Alpha: {lasso_cv.alpha_:.4f}")
print(f"Lasso CV Test R²: {r2_score(y_test, lasso_cv.predict(X_test_scaled)):.4f}")
print(f"Lasso CV Non-zero Features: {np.sum(lasso_cv.coef_ != 0)}")

# Plot Lasso cross-validation path
plt.figure(figsize=(10, 6))
plt.semilogx(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=1))
plt.axvline(x=lasso_cv.alpha_, color='red', linestyle='--', label=f'Best α={lasso_cv.alpha_:.4f}')
plt.xlabel('Alpha')
plt.ylabel('Mean Squared Error (CV)')
plt.title('Lasso Cross-Validation')
plt.legend()
plt.tight_layout()
plt.show()

## Expected Output & Interpretation

Regularization findings:

- **Ridge**: All coefficients shrunk but non-zero; best for multicollinearity
- **Lasso**: Some coefficients exactly zero; automatic feature selection
- **Cross-validation**: Automatically finds optimal regularization strength
- **Feature selection**: Lasso keeps alcohol, sulphates, volatile acidity as key predictors

## Hands-on Exercise 2.7

**Task**: Apply regularization to prevent overfitting.

1. Create polynomial features (degree 2) for the wine dataset
2. Train Linear, Ridge, and Lasso regression on these expanded features
3. Compare train and test R² for each model
4. How many features does Lasso select vs the total number?
5. Which model has the smallest gap between train and test R²?

**Time**: 15 minutes

In [ ]:
# TODO: degree-2 polynomial features; train Linear/Ridge/Lasso; compare train-test gaps; count Lasso nonzeros
# Hint: PolynomialFeatures(degree=2, include_bias=False); np.sum(model.coef_ != 0)


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

drop_cols = [c for c in ["quality", "good_wine"] if c in wine_df.columns]
X = wine_df.drop(columns=drop_cols); y = wine_df["quality"]

Xp = PolynomialFeatures(degree=2, include_bias=False).fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(Xp, y, test_size=0.2, random_state=42)
sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

for name, m in {"Linear": LinearRegression(),
                "Ridge": Ridge(alpha=1.0),
                "Lasso": Lasso(alpha=0.1, max_iter=10000)}.items():
    m.fit(X_tr, y_tr)
    tr = r2_score(y_tr, m.predict(X_tr)); te = r2_score(y_te, m.predict(X_te))
    extra = f", nonzero={np.sum(m.coef_ != 0)}/{len(m.coef_)}" if name == "Lasso" else ""
    print(f"{name}: train={tr:.3f} test={te:.3f} gap={tr - te:.3f}{extra}")
print("Lasso typically shows the smallest train-test gap (strongest regularization).")
```

</details>

# Module 2.8: Wow Moment — Performance Improvements

## Wow Factor: Before vs After Comparison

Let us compare our baseline to our best engineered and regularized model.

## Code: Complete Performance Summary

In [ ]:
# Build comprehensive comparison
np.random.seed(42)

# Original data
X_original = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

# Engineered features
wine_eng = wine_df.copy()
wine_eng['free_sulfur_ratio'] = wine_eng['free sulfur dioxide'] / (wine_eng['total sulfur dioxide'] + 0.001)
wine_eng['acid_ratio'] = wine_eng['fixed acidity'] / (wine_eng['volatile acidity'] + 0.001)
wine_eng['alcohol_density'] = wine_eng['alcohol'] * wine_eng['density']
wine_eng['log_chlorides'] = np.log1p(wine_eng['chlorides'])
X_engineered = wine_eng.drop(['quality', 'good_wine'], axis=1)

# Split
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_original, y, test_size=0.2, random_state=42
)
X_train_eng, X_test_eng, _, _ = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42
)

# Scale
scaler_orig = StandardScaler()
scaler_eng = StandardScaler()
X_train_orig_scaled = scaler_orig.fit_transform(X_train_orig)
X_test_orig_scaled = scaler_orig.transform(X_test_orig)
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

# Models to compare
comparison_results = []

# 1. Baseline Linear Regression
lr_baseline = LinearRegression()
lr_baseline.fit(X_train_orig_scaled, y_train)
comparison_results.append({
    'Model': 'Baseline Linear Regression',
    'Features': 'Original',
    'Train R²': r2_score(y_train, lr_baseline.predict(X_train_orig_scaled)),
    'Test R²': r2_score(y_test, lr_baseline.predict(X_test_orig_scaled)),
    'RMSE': np.sqrt(mean_squared_error(y_test, lr_baseline.predict(X_test_orig_scaled)))
})

# 2. Engineered Linear Regression
lr_eng = LinearRegression()
lr_eng.fit(X_train_eng_scaled, y_train)
comparison_results.append({
    'Model': 'Linear Regression',
    'Features': 'Engineered',
    'Train R²': r2_score(y_train, lr_eng.predict(X_train_eng_scaled)),
    'Test R²': r2_score(y_test, lr_eng.predict(X_test_eng_scaled)),
    'RMSE': np.sqrt(mean_squared_error(y_test, lr_eng.predict(X_test_eng_scaled)))
})

# 3. Ridge with engineered features
ridge = RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)
ridge.fit(X_train_eng_scaled, y_train)
comparison_results.append({
    'Model': 'Ridge Regression (CV)',
    'Features': 'Engineered',
    'Train R²': r2_score(y_train, ridge.predict(X_train_eng_scaled)),
    'Test R²': r2_score(y_test, ridge.predict(X_test_eng_scaled)),
    'RMSE': np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_eng_scaled)))
})

# 4. Lasso with engineered features
lasso = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
lasso.fit(X_train_eng_scaled, y_train)
comparison_results.append({
    'Model': 'Lasso Regression (CV)',
    'Features': 'Engineered',
    'Train R²': r2_score(y_train, lasso.predict(X_train_eng_scaled)),
    'Test R²': r2_score(y_test, lasso.predict(X_test_eng_scaled)),
    'RMSE': np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_eng_scaled)))
})

# Results table
results_df = pd.DataFrame(comparison_results)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)
print(results_df.to_string(index=False))

# Calculate improvement
baseline_r2 = comparison_results[0]['Test R²']
best_r2 = max([r['Test R²'] for r in comparison_results])
improvement = ((best_r2 - baseline_r2) / baseline_r2) * 100
print(f"\nImprovement from baseline to best: {improvement:.1f}%")

## Code: Visual Performance Summary

In [ ]:
# Create visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = results_df['Model'].tolist()
x = np.arange(len(models))

# Train vs Test R²
axes[0].bar(x - 0.2, results_df['Train R²'], 0.4, label='Train', alpha=0.8)
axes[0].bar(x + 0.2, results_df['Test R²'], 0.4, label='Test', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Train vs Test R² by Model')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Baseline\nLR', 'Engineered\nLR', 'Ridge\n(CV)', 'Lasso\n(CV)'], fontsize=9)
axes[0].legend()
axes[0].set_ylim(0, 0.5)

# Test R² comparison
colors = ['gray', 'steelblue', 'green', 'orange']
axes[1].bar(models, results_df['Test R²'], color=colors, edgecolor='black')
axes[1].axhline(y=baseline_r2, color='red', linestyle='--', label='Baseline')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Test R²')
axes[1].set_title('Test R² Comparison')
axes[1].set_xticklabels(['Baseline\nLR', 'Engineered\nLR', 'Ridge', 'Lasso'], fontsize=9)
axes[1].legend()

# RMSE comparison
axes[2].bar(models, results_df['RMSE'], color=colors, edgecolor='black')
axes[2].set_xlabel('Model')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Test RMSE Comparison (Lower is Better)')
axes[2].set_xticklabels(['Baseline\nLR', 'Engineered\nLR', 'Ridge', 'Lasso'], fontsize=9)

plt.tight_layout()
plt.show()

## Code: Key Takeaways Summary

In [ ]:
print("=" * 70)
print("PART 2 KEY ACHIEVEMENTS")
print("=" * 70)

print("\n1. FEATURE ENGINEERING IMPACT:")
print(f"   - Created 4 new domain-specific features")
print(f"   - R² improved from {comparison_results[0]['Test R²']:.4f} to {comparison_results[1]['Test R²']:.4f}")

print("\n2. REGULARIZATION BENEFITS:")
print(f"   - Ridge best alpha: {ridge.alpha_:.4f}")
print(f"   - Lasso best alpha: {lasso.alpha_:.4f}")
print(f"   - Lasso selected {np.sum(lasso.coef_ != 0)} of {len(lasso.coef_)} features")

print("\n3. OVERALL IMPROVEMENT:")
print(f"   - Baseline Test R²: {comparison_results[0]['Test R²']:.4f}")
print(f"   - Best Test R²: {best_r2:.4f}")
print(f"   - Relative improvement: {improvement:.1f}%")

print("\n4. TOP PREDICTIVE FEATURES (Lasso coefficients ≠ 0):")
lasso_features = pd.DataFrame({
    'Feature': X_engineered.columns,
    'Coefficient': lasso.coef_
})
lasso_features = lasso_features[lasso_features['Coefficient'] != 0].sort_values(
    'Coefficient', key=abs, ascending=False
)
for _, row in lasso_features.head(5).iterrows():
    print(f"   - {row['Feature']}: {row['Coefficient']:.4f}")

print("\n" + "=" * 70)
print("READY FOR ADVANCED MODELS (Part 3)")
print("=" * 70)

## Conclusion: Part 2

You have completed the second phase:

| Skill | Accomplishment |
|-------|----------------|
| Feature Engineering | Created impactful derived features |
| Feature Selection | Reduced features while maintaining performance |
| Train/Test Split | Implemented proper validation workflow |
| Overfitting Diagnosis | Identified and addressed overfitting |
| Linear Regression | Built interpretable prediction models |
| Logistic Regression | Achieved 75%+ classification accuracy |
| Regularization | Applied Ridge and Lasso with CV tuning |

Next session: **Advanced Models & Benchmarking**

## Part 2 Wrap-up Exercise

**Comprehensive Task**: Full modeling pipeline on California Housing.

1. Engineer at least 3 new features based on domain intuition
2. Apply feature selection using Lasso
3. Train Linear, Ridge, and Lasso regression models
4. Report train and test R² for all models
5. Create a bar chart comparing test performance
6. Write a brief (3-5 sentence) summary of which model performed best and why

**Time**: 30 minutes (homework if needed)

In [ ]:
# TODO: California Housing pipeline - engineer >=3 features, Linear/Ridge/Lasso, compare test R^2, bar chart, summary
# Hint: ratios like AveRooms/AveOccup make reasonable engineered features


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

df = fetch_california_housing(as_frame=True).frame.copy()

# 1. engineer 3 features
df["rooms_per_house"] = df["AveRooms"] / df["AveOccup"]
df["bedrooms_ratio"] = df["AveBedrms"] / df["AveRooms"]
df["income_per_room"] = df["MedInc"] / df["AveRooms"]

X = df.drop(columns=["MedHouseVal"]); y = df["MedHouseVal"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

results = {}
for name, m in {"Linear": LinearRegression(),
                "Ridge": Ridge(alpha=1.0),
                "Lasso": Lasso(alpha=0.01, max_iter=10000)}.items():
    m.fit(X_tr_s, y_tr)
    results[name] = (r2_score(y_tr, m.predict(X_tr_s)), r2_score(y_te, m.predict(X_te_s)))
    print(f"{name}: train R2={results[name][0]:.3f}, test R2={results[name][1]:.3f}")

plt.bar(results.keys(), [v[1] for v in results.values()], color=["gray", "green", "orange"])
plt.ylabel("Test R2"); plt.title("Model comparison"); plt.show()
print("Engineered ratios add a small lift; Ridge/Lasso match or slightly beat plain linear "
      "by controlling variance. Differences are modest because MedInc dominates and the "
      "relationship is largely linear.")
```

</details>

## End of Part 2

Continue to **Part 3: Advanced Models & Benchmarking**